## Headless Services — DNS to pod IPs, no virtual IP

Sometimes you don't want a virtual IP — you want clients to see **the individual pod IPs** and connect to a specific replica (or all of them). That's a **headless Service**, made by setting `clusterIP: None`:

```yaml
apiVersion: v1
kind: Service
metadata: { name: api }
spec:
  clusterIP: None      # the magic word — "headless"
  selector: { app: api }
  ports: [{ port: 80, targetPort: 5678 }]
```

Effects:

- **No ClusterIP, no kube-proxy rules** — there's no VIP to connect to.
- **DNS returns multiple A records** — one per Ready pod. A resolve-then-pick client (Java's resolver, Go's `net.LookupHost`) gets every pod IP.
- **EndpointSlices still exist** — the endpoints controller still tracks ready pods.

Two dominant use cases:

- **StatefulSets.** The canonical setup pairs a headless Service with a StatefulSet, giving **per-pod DNS**: `<pod>.<service>.<namespace>.svc.cluster.local`. That's how `cassandra-0.cassandra.default.svc.cluster.local` peers find each other (notebook 06).
- **Client-side load balancing.** gRPC, Kafka, connection-pool clients want to manage their own connection to every backend, not have kube-proxy round-robin behind their back. They look up the headless Service, learn every endpoint, and hold a long-lived connection to each.

**A regular `ClusterIP` is the right default** — headless is the building block under StatefulSets and a tool for specific client patterns, not a general upgrade. On our map it's still the **Service** chip, but resolving to *each* Pod individually rather than one shared VIP.